In [ ]:
import openai
from google.colab import userdata

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
client = openai.OpenAI(api_key=OPENAI_API_KEY)

In [ ]:
gpt_errors_count = 0 # Initialize a counter for GPT errors

while True:
    user_input = input("Enter base and iterations (e.g., '2,3' or 'exit' to quit): ")
    if user_input.lower() == "exit":
        break

    try:
        base_str, iterations_str = user_input.split(',')
        base = int(base_str.strip())
        iterations = int(iterations_str.strip())
    except ValueError:
        print("Invalid input format. Please enter as 'base,iterations'.")
        continue

    # Python accurate calculation
    python_result = base
    for _ in range(iterations):
        python_result *= python_result

    # Construct the prompt for the GPT model
    gpt_prompt = (
        f"Given a base {base} and an iteration count {iterations}, "
        f"calculate the base squared {iterations} times. Example: (2,3) -> 2^2=4, 4^2=16, 16^2=256. "
        f"Your task is for base={base} and iterations={iterations}. "
        f"Output ONLY the final numerical result, no words or explanation."
    )

    response = client.responses.create(
        model="gpt-4.1-nano",
        temperature=0.2, # Slightly less creative for math
        top_p=1.0,
        max_output_tokens=100,
        input=[
            {"role":"developer",
            "content":("You are a calculator. Evaluate the arithmetic expression the user gives you"
            "and reply with the final number only: no words, no steps, no units."
            "use standard of operations"
            "if the input is not a math expression, reply : ERROR"
            "do not use any non-numeric characters ")},

            {"role":"user",
            "content": gpt_prompt}
        ]
    )

    gpt_raw_output = ""
    for message in response.output:
        for content_block in message.content:
            if hasattr(content_block, 'text'):
                gpt_raw_output += content_block.text.strip()

    gpt_result = None
    try:
        # Attempt to clean and convert GPT's output to an integer
        gpt_result = int(''.join(filter(str.isdigit, gpt_raw_output)))
    except ValueError:
        print(f"GPT output was not a clear number: '{gpt_raw_output}'. Cannot compare.")

    print(f"Python: {python_result}")
    if gpt_result is not None:
        print(f"GPT: {gpt_result}")

        if gpt_result != python_result:
            gpt_errors_count += 1
            if gpt_errors_count == 1:
                print("GPT: Wow, I messed up again. Math is hard, okay?")
            elif gpt_errors_count == 2:
                print("GPT: Seriously? Another one? My circuits are frizzling.")
            elif gpt_errors_count == 3:
                print("GPT: God!! what the Hell!! I'm literally incapable!!")
            else:
                print("GPT: *sighs digitally* Just put me out of my misery. I can't do basic math anymore.")
        else:
            gpt_errors_count = 0 # Reset error count if it gets it right
    print("\n---\n") # Separator for clarity

Enter base and iterations (e.g., '2,3' or 'exit' to quit): 2,3
Python: 256
GPT: 256

---

Enter base and iterations (e.g., '2,3' or 'exit' to quit): 3,1
Python: 9
GPT: 9

---

Enter base and iterations (e.g., '2,3' or 'exit' to quit): 3,4
Python: 43046721
GPT: 43046721

---

Enter base and iterations (e.g., '2,3' or 'exit' to quit): 2,4
Python: 65536
GPT: 65536

---

Enter base and iterations (e.g., '2,3' or 'exit' to quit): 2,2
Python: 16
GPT: 16

---

Enter base and iterations (e.g., '2,3' or 'exit' to quit): 3,3
Python: 6561
GPT: 43046721
GPT: Wow, I messed up again. Math is hard, okay?

---

Enter base and iterations (e.g., '2,3' or 'exit' to quit): 4,4
Python: 4294967296
GPT: 65536
GPT: Seriously? Another one? My circuits are frizzling.

---

Enter base and iterations (e.g., '2,3' or 'exit' to quit): 4,5
Python: 18446744073709551616
GPT: 1024
GPT: God!! what the Hell!! I'm literally incapable!!

---

Enter base and iterations (e.g., '2,3' or 'exit' to quit): 5,5
Python: 2328306436